In [7]:
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import ( OrdinalEncoder, OneHotEncoder,LabelEncoder )
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler

from sklearn.naive_bayes import GaussianNB


from sklearn.model_selection import ( GridSearchCV,
RandomizedSearchCV,
StratifiedKFold,
cross_val_score)

from sklearn.metrics import (accuracy_score,precision_score, f1_score, recall_score , confusion_matrix, classification_report, ConfusionMatrixDisplay)


In [11]:
data = pd.read_csv('loan.csv')
data

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [13]:
data = data.drop('Loan_ID', axis=1)
data.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [15]:
le=LabelEncoder()
data['Loan_Status'] =le.fit_transform(data['Loan_Status'])
data.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,1
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,0
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,1
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,1
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,1


In [17]:
x = data.drop('Loan_Status',axis= 1)
y= data['Loan_Status']

x_train,x_test,y_train,y_test=train_test_split(x,y,random_state= 42 , test_size= 0.3)

In [19]:
num_cols = x.select_dtypes(include = ['int64','float64']).columns
cat_cols = x.select_dtypes(include = ['object']).columns

In [23]:
num_pipeline = Pipeline ([
    ('imputer',SimpleImputer (strategy = 'mean')),
    ('Scaler',StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy = 'most_frequent')),
    ('Encoding',OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num',num_pipeline,num_cols),
    ('cat',cat_pipeline,cat_cols)
])
 
pipe = Pipeline ([
    ('preprocessing', preprocessor ),
    ('model',GaussianNB())
])
pipe

pipe.fit(x_train,y_train)
pred = pipe.predict(x_test)
print("accuracy:", accuracy_score(y_test, pred))
print("pression:", precision_score(y_test, pred))
print("f1 score:", f1_score(y_test, pred))
print("recall score:", recall_score(y_test, pred))
print(classification_report (y_test, pred))
print(confusion_matrix(y_test,pred))

accuracy: 0.7837837837837838
pression: 0.7631578947368421
f1 score: 0.8529411764705882
recall score: 0.9666666666666667
              precision    recall  f1-score   support

           0       0.88      0.45      0.59        65
           1       0.76      0.97      0.85       120

    accuracy                           0.78       185
   macro avg       0.82      0.71      0.72       185
weighted avg       0.80      0.78      0.76       185

[[ 29  36]
 [  4 116]]


In [25]:
cv_score = cross_val_score (
    pipe, x, y, cv=5)
print(print("Cross Validation Scores :", cv_score))
print(cv_score.mean())

Cross Validation Scores : [0.78861789 0.74796748 0.76422764 0.80487805 0.80327869]
None
0.7817939490870318


In [27]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [29]:
param_grid = {
    'model__var_smoothing': [1e-9, 1e-8, 1e-7]
}
grid = GridSearchCV(
    estimator = pipe,
    param_grid = param_grid,
    cv=cv,
    n_jobs = 1
)
grid.fit(x_train,y_train)
#  RANDOMIZED SEARCH CV
print("\nBEST PARAMETERS")
print(grid.best_params_)

print("\nBEST SCORE")
print(grid.best_score_)


BEST PARAMETERS
{'model__var_smoothing': 1e-09}

BEST SCORE
0.7925581395348837


In [31]:
random_search = RandomizedSearchCV (
    estimator = pipe,
    param_distributions = param_grid ,
    cv= cv,
    n_iter = 50 )

random_search.fit(x_train,y_train)
print("\n Randomized excuted successfuly ")


print("|n BEST PARAMETERS ")
print(random_search.best_params_)


print("\n BEST SCORE ")
print(random_search.best_score_)

C:\Users\sowmy\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 3 is smaller than n_iter=50. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



 Randomized excuted successfuly 
|n BEST PARAMETERS 
{'model__var_smoothing': 1e-09}

 BEST SCORE 
0.7925581395348837
